# Figure 1: ensemble performance, pre vs post

- **Paper:** Fig 1
- **Original notebook:** `46_ensemble_performance_update.ipynb`
- **Reads:**
  - `outputs/24_, 26_, 27_, 42_, 44_, 45_pred_actual.pkl`
  - `outputs/49_pred_actual.pkl (pooled school model; extra plot only, not in Fig 1)`
- **Writes:**
  - `outputs/figures/Figure1.tiff`


In [ ]:
# Paths: edit config.py at the repo root, not this cell
import sys
from pathlib import Path
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(_root))
from config import DATA_DIR, DERIVED_DIR, OUTPUT_DIR, FIG_DIR

In [ ]:
# setup

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import random

# pip install optuna  # see requirements.txt
import optuna
from sklearn.metrics import root_mean_squared_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

import statsmodels.api as sm

from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import numpy as np
import PIL # not necessary but mustn't fail




In [ ]:
# read data
pre_school = pd.read_pickle(f"{OUTPUT_DIR}/24_pred_actual.pkl")
all_school = pd.read_pickle(f"{OUTPUT_DIR}/49_pred_actual.pkl")

pre_neighbor = pd.read_pickle(f"{OUTPUT_DIR}/26_pred_actual.pkl")
pre_both = pd.read_pickle(f"{OUTPUT_DIR}/27_pred_actual.pkl")

post_school = pd.read_pickle(f"{OUTPUT_DIR}/42_pred_actual.pkl")

post_neighbor = pd.read_pickle(f"{OUTPUT_DIR}/44_pred_actual.pkl")
post_both = pd.read_pickle(f"{OUTPUT_DIR}/45_pred_actual.pkl")




In [ ]:
from scipy.stats import linregress


In [ ]:
y_pred_36 = all_school.drop(columns = 'y_test').mean(axis=1)
y_actual_36 = all_school['y_test']

y_pred_24 = pre_school.drop(columns = 'y_test').mean(axis=1)
y_pred_26 = pre_neighbor.drop(columns = 'y_test').mean(axis=1)
y_pred_27 = pre_both.drop(columns = 'y_test').mean(axis=1)


y_pred_30 = post_school.drop(columns = 'y_test').mean(axis=1)
y_pred_31 = post_neighbor.drop(columns = 'y_test').mean(axis=1)
y_pred_32 = post_both.drop(columns = 'y_test').mean(axis=1)

y_actual_24 = pre_school['y_test']
y_actual_26 = pre_neighbor['y_test']
y_actual_27 = pre_both['y_test']

y_actual_30 = post_school['y_test']
y_actual_31 = post_neighbor['y_test']
y_actual_32 = post_both['y_test']

In [ ]:
subplot_size = 5  # Size of each subplot in inches
fig_width = subplot_size * 3
fig_height = subplot_size * 2

# Pre-COVID and Post-COVID Plot (School, Neighborhood, Both)
fig, axs = plt.subplots(2, 3, figsize=(fig_width, fig_height))
axs = axs.flatten()
# School (Pre-COVID)
r2_pre_school = r2_score(y_actual_24, y_pred_24)
rmse_pre_school = root_mean_squared_error(y_actual_24, y_pred_24)

axs[0].scatter(y_pred_24, y_actual_24, color='k', alpha=0.5)
axs[0].set_title('(a) Pre-Covid School',fontsize = 20)
axs[0].set_xlabel('Predicted',fontsize = 14)
axs[0].set_ylabel('Actual',fontsize = 14)
axs[0].set_xlim((0, 100))
axs[0].set_ylim((0, 100))
axs[0].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_24, y_actual_24)
axs[0].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[0].text(72.5, 15,f"$R^2 = {r2_pre_school:.2f}$",fontsize = 18)
axs[0].text(63, 5,f"$RMSE = {rmse_pre_school:.2f}$",fontsize = 18)

axs[0].tick_params(axis='both', labelsize=12)

# Neighborhood (Pre-COVID)
r2_pre_neighbor = r2_score(y_actual_26, y_pred_26)
rmse_pre_neighbor = root_mean_squared_error(y_actual_26, y_pred_26)

axs[1].scatter(y_pred_26, y_actual_26, color='k', alpha=0.5)
axs[1].set_title('(b) Pre-Covid Neighborhood',fontsize = 20)
axs[1].set_xlabel('Predicted',fontsize = 14)
axs[1].set_ylabel('')
axs[1].set_xlim((0, 100))
axs[1].set_ylim((0, 100))
axs[1].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_26, y_actual_26)
axs[1].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[1].text(72.5, 15, f"$R^2 = {r2_pre_neighbor:.2f}$",fontsize = 18)
axs[1].text(60, 5,f"$RMSE = {rmse_pre_neighbor:.2f}$",fontsize = 18)

axs[1].tick_params(left=False, labelleft=False)


# Both (Pre-COVID)
r2_pre_both = r2_score(y_actual_27, y_pred_27)
rmse_pre_both = root_mean_squared_error(y_actual_27, y_pred_27)

axs[2].scatter(y_pred_27, y_actual_27, color='k', alpha=0.5)
axs[2].set_title('(c) Pre-Covid Both',fontsize = 20)
axs[2].set_xlabel('Predicted',fontsize = 14)
axs[2].set_ylabel("")
axs[2].set_xlim((0, 100))
axs[2].set_ylim((0, 100))
axs[2].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_27, y_actual_27)
axs[2].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[2].text(72.5, 15, f"$R^2 = {r2_pre_both:.2f}$",fontsize = 18)
axs[2].text(63, 5,f"$RMSE = {rmse_pre_both:.2f}$",fontsize = 18)

axs[2].tick_params(left=False, labelleft=False)



# Post-COVID Plot (School, Neighborhood, Both)

# School (Post-COVID)
r2_post_school = r2_score(y_actual_30, y_pred_30)
rmse_post_school = root_mean_squared_error(y_actual_30, y_pred_30)

axs[3].scatter(y_pred_30, y_actual_30, color='k', alpha=0.5)
axs[3].set_title('(d) During Covid School',fontsize = 20)
axs[3].set_xlabel('Predicted',fontsize = 14)
axs[3].set_ylabel('Actual',fontsize = 14)
axs[3].set_xlim((0, 100))
axs[3].set_ylim((0, 100))
axs[3].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_30, y_actual_30)
axs[3].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[3].text(72.5, 15,  f"$R^2 = {r2_post_school:.2f}$",fontsize = 18)
axs[3].text(63, 5,f"$RMSE = {rmse_post_school:.2f}$",fontsize = 18)

axs[3].tick_params(axis='both', labelsize=12)

# Neighborhood (Post-COVID)
r2_post_neighbor = r2_score(y_actual_31, y_pred_31)
rmse_post_neighbor = root_mean_squared_error(y_actual_31, y_pred_31)

axs[4].scatter(y_pred_31, y_actual_31, color='k', alpha=0.5)
axs[4].set_title('(e) During Covid Neighborhood',fontsize = 20)
axs[4].set_xlabel('Predicted',fontsize = 14)
axs[4].set_ylabel('')
axs[4].set_xlim((0, 100))
axs[4].set_ylim((0, 100))
axs[4].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_31, y_actual_31)
axs[4].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[4].text(72.5, 15, f"$R^2 = {r2_post_neighbor:.2f}$",fontsize = 18)
axs[4].text(60, 5,f"$RMSE = {rmse_post_neighbor:.2f}$",fontsize = 18)

axs[4].tick_params(left=False, labelleft=False)


# Both (Post-COVID)
r2_post_both = r2_score(y_actual_32, y_pred_32)
rmse_post_both = root_mean_squared_error(y_actual_32, y_pred_32)

axs[5].scatter(y_pred_32, y_actual_32, color='k', alpha=0.5)
axs[5].set_title('(f) During Covid Both',fontsize = 20)
axs[5].set_xlabel('Predicted',fontsize = 14)
axs[5].set_ylabel("")
axs[5].set_xlim((0, 100))
axs[5].set_ylim((0, 100))
axs[5].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_32, y_actual_32)
axs[5].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[5].text(72.5, 15, f"$R^2 = {r2_post_both:.2f}$",fontsize = 18)
axs[5].text(60, 5,f"$RMSE = {rmse_post_both:.2f}$",fontsize = 18)

axs[5].tick_params(left=False, labelleft=False)

plt.subplots_adjust(wspace=-.1)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/Figure1.tiff',dpi=600,format="tiff", pil_kwargs={"compression": "tiff_lzw"})

plt.show()


In [ ]:



r2_overall = r2_score(y_actual_36, y_pred_36)

# plot
fig, ax = plt.subplots(figsize=(6, 6)) # fig dim
ax.scatter(y_pred_36, y_actual_36, color='k', alpha=0.5)
ax.set_ylabel('Actual College Going Rate (%)')
ax.set_xlabel('Predicted College Going Rate (%)')
ax.set_title('School Only — Actual vs. Predicted Values')
ax.text(80, 5, f"$R^2 = {r2_overall:.3f}$")
ax.set_ylim((0, 100))
ax.set_xlim((0, 100))

xpoints = ypoints = ax.get_xlim()
ax.plot(xpoints, ypoints, linestyle='--',alpha =.75 ,color='grey', lw=2, scalex=False, scaley=False, label="y = x")


slope, intercept, r_value, p_value, std_err = linregress(x = y_pred_36, y= y_actual_36)
ax.plot(xpoints, slope * np.array(xpoints) + intercept,alpha=0.7 ,color='red', lw=2, label='Actual ~ Predicted')

ax.legend()
plt.show()


In [ ]:
plt.scatter(y_actual_24, y_pred_24,color = 'k')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Pre-Covid School — Predicted vs. Actual Values')
plt.ylim((0,100))
plt.xlim((0,100))
xpoints = ypoints = plt.xlim()
plt.plot(xpoints, ypoints, linestyle='-', color='r', lw=2, scalex=False,
         scaley=False,label = "y = x")
plt.legend()
plt.show()

In [ ]:
plt.scatter(y_actual_26, y_pred_26,color = 'k')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Pre-Covid Neighborhood — Predicted vs. Actual Values')
plt.ylim((0,1))
plt.xlim((0,1))
xpoints = ypoints = plt.xlim()
plt.plot(xpoints, ypoints, linestyle='-', color='r', lw=2, scalex=False,
         scaley=False,label = "y = x")
plt.legend()

plt.show()

In [ ]:
plt.scatter(y_actual_27, y_pred_27,color = 'k')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Pre-Covid Both — Predicted vs. Actual Values')
plt.ylim((0,1))
plt.xlim((0,1))
xpoints = ypoints = plt.xlim()
plt.plot(xpoints, ypoints, linestyle='-', color='r', lw=2, scalex=False,
         scaley=False,label = "y = x")
plt.legend()

plt.show()

In [ ]:
plt.scatter(y_actual_30, y_pred_30,color = 'k')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Post-Covid School — Predicted vs. Actual Values')
plt.ylim((0,100))
plt.xlim((0,100))
xpoints = ypoints = plt.xlim()
plt.plot(xpoints, ypoints, linestyle='-', color='r', lw=2, scalex=False,
         scaley=False,label = "y = x")
plt.legend()

plt.show()

In [ ]:
plt.scatter(y_actual_31, y_pred_31,color = 'k')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Post-Covid Neighborhood — Predicted vs. Actual Values')
plt.ylim((0,1))
plt.xlim((0,1))
xpoints = ypoints = plt.xlim()
plt.plot(xpoints, ypoints, linestyle='-', color='r', lw=2, scalex=False,
         scaley=False,label = "y = x")
plt.legend()

plt.show()

In [ ]:
plt.scatter(y_actual_32, y_pred_32,color = 'k')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Post-Covid Both — Predicted vs. Actual Values')
plt.ylim((0,1))
plt.xlim((0,1))
xpoints = ypoints = plt.xlim()
plt.plot(xpoints, ypoints, linestyle='-', color='r', lw=2, scalex=False,
         scaley=False,label = "y = x")
plt.legend()

plt.show()

In [ ]:
# Pre-COVID Plot (School, Neighborhood, Both)
fig, axs = plt.subplots(1, 3, figsize=(18, 6)) # 1 row, 3 columns

# School (Pre-COVID)
r2_pre_school = r2_score(y_actual_24, y_pred_24)
axs[0].scatter(y_pred_24, y_actual_24, color='k', alpha=0.5)
axs[0].set_title('Pre-Covid School',fontsize = 16)
axs[0].set_xlabel('Predicted',fontsize = 14)
axs[0].set_ylabel('Actual',fontsize = 14)
axs[0].set_xlim((0, 100))
axs[0].set_ylim((0, 100))
axs[0].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_24, y_actual_24)
axs[0].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[0].text(80, 5, f"$R^2 = {r2_pre_school:.3f}$",fontsize = 14)
axs[0].tick_params(axis='both', labelsize=12)

# Neighborhood (Pre-COVID)
r2_pre_neighbor = r2_score(y_actual_26, y_pred_26)
axs[1].scatter(y_pred_26, y_actual_26, color='k', alpha=0.5)
axs[1].set_title('Pre-Covid Neighborhood',fontsize = 16)
axs[1].set_xlabel('Predicted',fontsize = 14)
axs[1].set_ylabel('')
axs[1].set_xlim((0, 100))
axs[1].set_ylim((0, 100))
axs[1].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_26, y_actual_26)
axs[1].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[1].text(80, 5, f"$R^2 = {r2_pre_neighbor:.3f}$",fontsize = 14)
axs[1].tick_params(left=False, labelleft=False)


# Both (Pre-COVID)
r2_pre_both = r2_score(y_actual_27, y_pred_27)
axs[2].scatter(y_pred_27, y_actual_27, color='k', alpha=0.5)
axs[2].set_title('Pre-Covid Both',fontsize = 16)
axs[2].set_xlabel('Predicted',fontsize = 14)
axs[2].set_ylabel("")
axs[2].set_xlim((0, 100))
axs[2].set_ylim((0, 100))
axs[2].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_27, y_actual_27)
axs[2].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[2].text(80, 5, f"$R^2 = {r2_pre_both:.3f}$",fontsize = 14)
axs[2].tick_params(left=False, labelleft=False)

plt.subplots_adjust(wspace=-.1)

plt.tight_layout()
plt.show()


In [ ]:
# Pre-COVID Plot (School, Neighborhood, Both)
fig, axs = plt.subplots(1, 3, figsize=(18, 6)) # 1 row, 3 columns

# School (Post-COVID)
r2_post_school = r2_score(y_actual_30, y_pred_30)
axs[0].scatter(y_pred_30, y_actual_30, color='k', alpha=0.5)
axs[0].set_title('Post-Covid School',fontsize = 16)
axs[0].set_xlabel('Predicted',fontsize = 14)
axs[0].set_ylabel('Actual',fontsize = 14)
axs[0].set_xlim((0, 100))
axs[0].set_ylim((0, 100))
axs[0].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_30, y_actual_30)
axs[0].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[0].text(80, 5, f"$R^2 = {r2_post_school:.3f}$",fontsize = 14)
axs[0].tick_params(axis='both', labelsize=12)

# Neighborhood (Post-COVID)
r2_post_neighbor = r2_score(y_actual_31, y_pred_31)
axs[1].scatter(y_pred_31, y_actual_31, color='k', alpha=0.5)
axs[1].set_title('Post-Covid Neighborhood',fontsize = 16)
axs[1].set_xlabel('Predicted',fontsize = 14)
axs[1].set_ylabel('')
axs[1].set_xlim((0, 100))
axs[1].set_ylim((0, 100))
axs[1].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_31, y_actual_31)
axs[1].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[1].text(80, 5, f"$R^2 = {r2_post_neighbor:.3f}$",fontsize = 14)
axs[1].tick_params(left=False, labelleft=False)


# Both (Pre-COVID)
r2_post_both = r2_score(y_actual_32, y_pred_32)
axs[2].scatter(y_pred_32, y_actual_32, color='k', alpha=0.5)
axs[2].set_title('Post-Covid Both',fontsize = 16)
axs[2].set_xlabel('Predicted',fontsize = 14)
axs[2].set_ylabel("")
axs[2].set_xlim((0, 100))
axs[2].set_ylim((0, 100))
axs[2].plot([0, 100], [0, 100], linestyle='--', color='grey', lw=2)  # y=x line
slope, intercept, _, _, _ = linregress(y_pred_32, y_actual_32)
axs[2].plot([0, 100], [slope*0 + intercept, slope*100 + intercept], color='red', lw=2)  # regression line
axs[2].text(80, 5, f"$R^2 = {r2_post_both:.3f}$",fontsize = 14)
axs[2].tick_params(left=False, labelleft=False)

plt.subplots_adjust(wspace=-.1)

plt.tight_layout()
plt.show()
